# CreditWise Loan Approval Analysis

This notebook explores the bundled loan dataset and evaluates the same leakage-safe machine-learning pipeline used by the Streamlit application. The model is a decision-support demonstration, not an automated lending decision system.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.model import (
    DEFAULT_DATA_PATH, evaluate_model, feature_importance,
    load_data, train_production_model
)

sns.set_theme(style="whitegrid")
raw = pd.read_csv(DEFAULT_DATA_PATH)
raw.head()

## 1. Data quality
Rows with a missing target are excluded. Feature imputation happens inside the model pipeline after each train/test split, preventing information from the test set leaking into training.

In [ ]:
quality = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(1),
    "unique": raw.nunique(dropna=True),
})
print(f"Raw rows: {len(raw):,}")
print(f"Labelled rows: {raw['Loan_Approved'].notna().sum():,}")
quality

## 2. Exploratory analysis

In [ ]:
labelled = raw.dropna(subset=["Loan_Approved"]).copy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=labelled, x="Loan_Approved", hue="Loan_Approved", legend=False, ax=axes[0])
axes[0].set_title("Target distribution")
sns.boxplot(data=labelled, x="Loan_Approved", y="Credit_Score", ax=axes[1])
axes[1].set_title("Credit score by outcome")
sns.boxplot(data=labelled, x="Loan_Approved", y="DTI_Ratio", ax=axes[2])
axes[2].set_title("Debt-to-income ratio by outcome")
plt.tight_layout()
plt.show()

In [ ]:
numeric = labelled.select_dtypes(include="number").copy()
numeric["Loan_Approved"] = labelled["Loan_Approved"].map({"No": 0, "Yes": 1})
correlations = numeric.corr(numeric_only=True)["Loan_Approved"].drop("Loan_Approved").sort_values()
correlations.plot(kind="barh", figsize=(9, 6), title="Numeric feature correlation with approval")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## 3. Reproducible model evaluation
The holdout test uses a stratified 80/20 split with random_state=42. Five-fold stratified cross-validation gives a more stable estimate across multiple splits. Applicant ID, gender, and marital status are excluded from prediction.

In [ ]:
features, target = load_data()
metrics = evaluate_model(features, target)

holdout = pd.Series(metrics["holdout"]).drop("confusion_matrix")
cross_validation = pd.DataFrame(metrics["cross_validation"]).T
display(holdout.to_frame("value"))
display(cross_validation)
print("Confusion matrix [[TN, FP], [FN, TP]]:", metrics["holdout"]["confusion_matrix"])

## 4. Final model and explainability
After evaluation, the application model is refitted on every labelled row. Feature importance reflects model usage, not causality.

In [ ]:
production_model = train_production_model(features, target)
importance = feature_importance(production_model, limit=12)
sns.barplot(data=importance, x="importance", y="feature")
plt.title("Top model feature importances")
plt.tight_layout()
plt.show()
importance